# Knowledge Tracing: per-student topic mastery (BKT)

Fits a **BKT** (Bayesian Knowledge Tracing) model over this project's student interaction
graph to answer the actual product question: *for a given student, which topics are they
weak in?*

BKT is a 2-state HMM per (student, leaf topic), updated online per attempt — cheap,
interpretable, and it needs only a handful of attempts per pair to produce a defensible
`p_know`, including graceful cold-start behavior (falls back to the prior) for a new
student or a just-published topic. Its output is directly the personalization signal
this system needs: one `p_know` per (student, topic), so "what is this student weak in"
is just "sort their topics by `p_know` ascending."

Data: `scripts/generate_dummy_interactions.py`'s synthetic students, generated against
the real question bank's subject/topic taxonomy (`notebooks/mcq_output/question_bank.json`)
so topic-level sequences here have the same shape production data will have. Current scale:
20 students, ~1100 quiz answers, 56 simulated topics.

Requires `make neo4j-up` and the dummy data already generated (see
`docs/07-operations.md` / this repo's README).

In [1]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import roc_auc_score, log_loss

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.student_kg.driver import make_driver

pd.set_option("display.max_rows", 20)
RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

## 1. Pull interaction sequences from Neo4j

One row per `QUIZ_ANSWER` event, joined to the leaf `Topic` it's tagged with via
`(:Question)-[:BELONGS_TO]->(:Topic)` (see `src/quiz/attempts.py::record_attempt` — the
leaf topic is the deepest element of `topic_tag`). Ordered by `ts` within each student so
sequence models see the real chronological order.

In [2]:
_SEQUENCES_QUERY = """
MATCH (s:Student)-[:ATTEMPTED]->(:QuizSession)-[:HAS_ANSWER]->(e:InteractionEvent {type: "QUIZ_ANSWER"})
MATCH (e)-[:FOR_QUESTION]->(q:Question)-[:BELONGS_TO]->(t:Topic)
OPTIONAL MATCH (s)-[r:REVIEWING]->(q)
RETURN s.id AS student_id, t.path AS topic_path, e.question_uid AS question_uid,
       e.correct AS correct, e.confidence AS confidence, e.ts AS ts,
       r.attempt_count AS attempt_count
ORDER BY s.id, e.ts ASC
"""

driver = make_driver()
with driver.session() as session:
    records = [dict(r) for r in session.run(_SEQUENCES_QUERY)]
driver.close()

df = pd.DataFrame(records)
df["ts"] = df["ts"].apply(lambda z: z.to_native())
df["correct"] = df["correct"].astype(int)
df["confidence"] = df["confidence"].fillna("unsure")
print(f"{len(df)} attempts, {df.student_id.nunique()} students, {df.topic_path.nunique()} topics")
print(df["confidence"].value_counts())
df.head()

10480 attempts, 43 students, 56 topics
confidence
unsure       3682
guessing     3629
confident    3169
Name: count, dtype: int64


,student_id,topic_path,question_uid,correct,confidence,ts,attempt_count
0,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Bile ...,master_mcq::copy_of_paket_a49_pdf::0003,1,confident,2026-03-09 00:01:46.091792+00:00,1
1,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Bile ...,master_mcq::paket_8_pdf::0001,1,unsure,2026-03-09 00:03:47.575590+00:00,3
2,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Bile ...,master_mcq::paket_49_pdf::0007,1,confident,2026-03-09 00:05:45.508108+00:00,2
3,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Bile ...,master_mcq::paket_8_pdf::0001,1,confident,2026-03-09 00:08:47.195265+00:00,3
4,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Bile ...,master_mcq::paket_8_pdf::0001,1,unsure,2026-03-09 00:09:56.205623+00:00,3


In [3]:
# Data-scale diagnostics up front — these numbers are what make the BKT-vs-AKT call later.
attempts_per_student_topic = df.groupby(["student_id", "topic_path"]).size()
print("attempts per (student, topic) pair:")
print(attempts_per_student_topic.describe())
print()
print("median attempts/student/topic:", attempts_per_student_topic.median())
print("pairs with >=5 attempts:", (attempts_per_student_topic >= 5).sum(),
      "/", len(attempts_per_student_topic))

attempts per (student, topic) pair:
count    1554.000000
mean        6.743887
std         2.266823
min         3.000000
25%         6.000000
50%         6.000000
75%         6.000000
max        24.000000
dtype: float64

median attempts/student/topic: 6.0
pairs with >=5 attempts: 1509 / 1554


## 2. BKT baseline (confidence-conditional emissions)

Standard 2-state (knows / doesn't-know) HMM per (student, topic):

- `p_init` — prior P(knows) before any evidence.
- `p_transit` — P(learns) between an unknown state and the next attempt.
- `p_slip` — P(wrong | knows).
- `p_guess` — P(correct | doesn't know).

Every attempt already carries a self-reported `confidence` label (`confident` / `unsure` /
`guessing` — see `src/quiz/attempts.py::record_attempt`'s docstring: "a correct guess and a
confident correct answer are not the same evidence of mastery"). Plain BKT throws that
signal away and treats every correct/wrong answer as equally strong evidence. Instead we
make `p_slip` and `p_guess` **conditional on the reported confidence**, so the same
correct/wrong outcome updates `p_know` by a different amount depending on how the student
said they got there:

- `confident` — low `p_slip`, low `p_guess`: a confident-correct answer is trusted as real
  evidence of knowing; a confident-wrong answer is trusted as real evidence of a
  misconception (not a slip), so it pulls `p_know` down hard.
- `guessing` — high `p_slip`, high `p_guess`: outcomes are close to a coin flip either way,
  so both correct and wrong answers move `p_know` only a little — a lucky guess shouldn't
  look like mastery, and a wrong guess shouldn't look like a confirmed gap.
- `unsure` — in between the two, close to the original flat defaults.

Update rule (per attempt, params now indexed by that attempt's `confidence`):

```
p_slip, p_guess = P_SLIP[confidence], P_GUESS[confidence]

if observed correct:
    p_know_post = p_know * (1 - p_slip) / (p_know * (1 - p_slip) + (1 - p_know) * p_guess)
else:
    p_know_post = p_know * p_slip / (p_know * p_slip + (1 - p_know) * (1 - p_guess))

p_know_next = p_know_post + (1 - p_know_post) * p_transit
```

`p_init`/`p_transit` stay global scalars — only the emission probabilities (`p_slip`,
`p_guess`) vary by confidence. All params are still fixed defaults (not EM-fit per topic);
EM-fitting per topic, and validating the confidence buckets actually separate accuracy the
way we're assuming (guessing ≈ chance rate, confident ≫ unsure), is follow-up work.

In [4]:
BKT_PARAMS = dict(
    p_init=0.3,
    p_transit=0.1,
    p_slip={"confident": 0.05, "unsure": 0.10, "guessing": 0.30},
    p_guess={"confident": 0.10, "unsure": 0.25, "guessing": 0.50},
)


def bkt_update(p_know: float, correct: bool, confidence: str, params: dict) -> float:
    p_slip = params["p_slip"][confidence]
    p_guess = params["p_guess"][confidence]
    p_transit = params["p_transit"]
    if correct:
        num = p_know * (1 - p_slip)
        denom = num + (1 - p_know) * p_guess
    else:
        num = p_know * p_slip
        denom = num + (1 - p_know) * (1 - p_guess)
    p_know_post = num / denom if denom > 0 else p_know
    return p_know_post + (1 - p_know_post) * p_transit


def bkt_predict_proba(p_know: float, confidence: str, params: dict) -> float:
    """P(correct) implied by current p_know, before seeing the observation. Needs
    `confidence` too since p_slip/p_guess are now confidence-conditional — this is a
    genuine next-step prediction only in hindsight-eval mode, where the label is known
    ahead of time from logged data; a live recommender doesn't know the student's
    confidence before they answer, so it would predict off the `unsure` (middle) row
    instead."""
    p_slip = params["p_slip"][confidence]
    p_guess = params["p_guess"][confidence]
    return p_know * (1 - p_slip) + (1 - p_know) * p_guess


def run_bkt(df: pd.DataFrame, params: dict = BKT_PARAMS) -> tuple[pd.DataFrame, dict]:
    """Runs BKT forward over every (student, topic) sequence in chronological order.
    Returns per-attempt predicted P(correct) (predicted BEFORE the update, i.e. a genuine
    next-step prediction, not a fitted-in-hindsight one) and the final p_know per pair."""
    preds = np.empty(len(df))
    state: dict[tuple[str, str], float] = {}
    for i, row in enumerate(df.itertuples()):
        key = (row.student_id, row.topic_path)
        p_know = state.get(key, params["p_init"])
        preds[i] = bkt_predict_proba(p_know, row.confidence, params)
        state[key] = bkt_update(p_know, bool(row.correct), row.confidence, params)
    out = df.copy()
    out["bkt_pred"] = preds
    return out, state


df_sorted = df.sort_values(["student_id", "topic_path", "ts"]).reset_index(drop=True)
bkt_results, bkt_final_state = run_bkt(df_sorted)
bkt_results[["student_id", "topic_path", "ts", "correct", "confidence", "bkt_pred"]].head(10)

,student_id,topic_path,ts,correct,confidence,bkt_pred
0,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Bile ...,2026-03-09 00:01:46.091792+00:00,1,confident,0.355000
1,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Bile ...,2026-03-09 00:03:47.575590+00:00,1,unsure,0.784648
2,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Bile ...,2026-03-09 00:05:45.508108+00:00,1,confident,0.906745
3,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Bile ...,2026-03-09 00:08:47.195265+00:00,1,confident,0.945707
4,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Bile ...,2026-03-09 00:09:56.205623+00:00,1,unsure,0.899688
5,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Bile ...,2026-03-09 00:11:19.891522+00:00,1,confident,0.949898
6,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Liver...,2026-03-10 19:12:34.735247+00:00,1,guessing,0.560000
7,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Liver...,2026-03-10 19:15:57.951452+00:00,1,unsure,0.534375
8,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Liver...,2026-03-10 19:17:50.196635+00:00,0,unsure,0.746053
9,088a0ba8-56d7-44ed-9116-79ec03fd3453,Abdominal wall - hollow organ - organs > Liver...,2026-03-10 19:21:04.044216+00:00,0,unsure,0.490803


In [5]:
bkt_auc = roc_auc_score(bkt_results["correct"], bkt_results["bkt_pred"])
bkt_ll = log_loss(bkt_results["correct"], bkt_results["bkt_pred"].clip(1e-4, 1 - 1e-4))
print(f"BKT next-step prediction — AUC: {bkt_auc:.4f}, log loss: {bkt_ll:.4f}")

BKT next-step prediction — AUC: 0.5658, log loss: 0.7491


In [6]:
# Sanity check on the P_SLIP/P_GUESS assumption baked into BKT_PARAMS: raw accuracy
# should actually separate by confidence bucket (guessing near chance rate, confident
# well above it) before trusting the conditional emission model over the flat one.
print(bkt_results.groupby("confidence")["correct"].agg(["mean", "count"]))

                mean  count
confidence                 
confident   0.712212   3169
guessing    0.326261   3629
unsure      0.534221   3682


## 2b. EM-fitting BKT_PARAMS (Baum-Welch)

The 0.5658 AUC above comes from `BKT_PARAMS` picked by hand, never fit to data — the
cell 4 docstring flagged this as follow-up work. Fit `p_init`, `p_transit`, and
confidence-conditional `p_slip`/`p_guess` by EM (Baum-Welch: forward-backward E-step,
closed-form 2-state M-step) instead.

Fit **globally** (one set of params, pooled across every (student, topic) sequence),
not per-topic. Per-topic EM needs the ~25-students-per-topic floor from Slater & Baker
(2018) to converge — cell 3 shows 43 students spread across 56 topics, well under that
per-topic, so a per-topic fit here would reproduce the exact degenerate-parameter
failure mode that paper measured. Pooling all sequences into one fit stays inside the
regime EM is actually reliable in; per-topic fitting is future work once there's more
students/topic.

Each attempt's `confidence` selects which `p_slip[c]`/`p_guess[c]` pair generated it, so
the E-step's emission probability is looked up per-observation by its logged
`confidence` rather than assuming one emission distribution for the whole sequence.

In [7]:
CONFIDENCE_LEVELS = ("confident", "unsure", "guessing")


def _build_sequences(df: pd.DataFrame) -> list[list[tuple[int, str]]]:
    """One (correct, confidence) list per (student, topic) pair, in chronological order —
    the unit Baum-Welch runs forward-backward over."""
    sequences = []
    for _, group in df.groupby(["student_id", "topic_path"], sort=False):
        sequences.append(list(zip(group["correct"].astype(int), group["confidence"])))
    return sequences


def _emission_prob(correct: int, confidence: str, state: int, params: dict) -> float:
    """P(observed correct | hidden state), state 0 = doesn't-know, state 1 = knows."""
    p_slip = params["p_slip"][confidence]
    p_guess = params["p_guess"][confidence]
    if state == 1:
        return (1 - p_slip) if correct else p_slip
    return p_guess if correct else (1 - p_guess)


def _forward_backward(seq: list[tuple[int, str]], params: dict):
    """Standard 2-state forward-backward. Transition matrix is the BKT one-directional
    learning assumption (state 0 -> 1 w.p. p_transit, no forgetting: 1 -> 1 always)."""
    T = len(seq)
    p_init, p_transit = params["p_init"], params["p_transit"]
    trans = np.array([[1 - p_transit, p_transit], [0.0, 1.0]])

    emit = np.empty((T, 2))
    for t, (correct, conf) in enumerate(seq):
        emit[t, 0] = _emission_prob(correct, conf, 0, params)
        emit[t, 1] = _emission_prob(correct, conf, 1, params)

    alpha = np.empty((T, 2))
    scale = np.empty(T)
    alpha[0] = np.array([1 - p_init, p_init]) * emit[0]
    scale[0] = alpha[0].sum()
    alpha[0] /= scale[0]
    for t in range(1, T):
        alpha[t] = (alpha[t - 1] @ trans) * emit[t]
        scale[t] = alpha[t].sum()
        alpha[t] /= scale[t]

    beta = np.empty((T, 2))
    beta[T - 1] = 1.0
    for t in range(T - 2, -1, -1):
        beta[t] = (trans @ (emit[t + 1] * beta[t + 1])) / scale[t + 1]

    gamma = alpha * beta
    gamma /= gamma.sum(axis=1, keepdims=True)

    # xi[t] = P(state_t=0, state_{t+1}=1 | obs) — the only nonzero transition besides
    # self-loops, since state 1 -> 0 has zero probability by construction.
    xi01 = np.empty(T - 1)
    for t in range(T - 1):
        xi01[t] = (
            alpha[t, 0] * p_transit * emit[t + 1, 1] * beta[t + 1, 1]
        ) / scale[t + 1]

    loglik = np.log(scale).sum()
    return gamma, xi01, loglik


def fit_bkt_em(
    sequences: list[list[tuple[int, str]]],
    init_params: dict,
    n_iter: int = 20,
    tol: float = 1e-4,
) -> tuple[dict, list[float]]:
    """Baum-Welch EM, pooled across all sequences (one shared param set). Each M-step
    aggregates expected counts over every sequence before updating params, so this is a
    genuine joint fit, not per-sequence fits averaged after the fact."""
    params = {
        "p_init": init_params["p_init"],
        "p_transit": init_params["p_transit"],
        "p_slip": dict(init_params["p_slip"]),
        "p_guess": dict(init_params["p_guess"]),
    }
    history = []

    for it in range(n_iter):
        init_num = 0.0
        trans_num, trans_denom = 0.0, 0.0
        slip_num = {c: 0.0 for c in CONFIDENCE_LEVELS}
        slip_denom = {c: 0.0 for c in CONFIDENCE_LEVELS}
        guess_num = {c: 0.0 for c in CONFIDENCE_LEVELS}
        guess_denom = {c: 0.0 for c in CONFIDENCE_LEVELS}
        total_loglik = 0.0

        for seq in sequences:
            gamma, xi01, loglik = _forward_backward(seq, params)
            total_loglik += loglik

            init_num += gamma[0, 1]
            trans_num += xi01.sum()
            trans_denom += gamma[:-1, 0].sum()

            for t, (correct, conf) in enumerate(seq):
                p_know_state, p_unknow_state = gamma[t, 1], gamma[t, 0]
                if correct:
                    slip_num[conf] += p_know_state  # wrong: (1-correct)*p_know
                    guess_num[conf] += p_unknow_state  # right & unknown
                slip_denom[conf] += p_know_state
                guess_denom[conf] += p_unknow_state

        n_seq = len(sequences)
        params["p_init"] = np.clip(init_num / n_seq, 1e-3, 1 - 1e-3)
        params["p_transit"] = np.clip(
            trans_num / trans_denom if trans_denom > 0 else params["p_transit"],
            1e-3,
            1 - 1e-3,
        )
        for c in CONFIDENCE_LEVELS:
            # p_slip = P(wrong | knows) -> slip_num accumulated P(correct & knows), so
            # slip = 1 - correct_given_know_rate.
            correct_given_know = slip_num[c] / slip_denom[c] if slip_denom[c] > 0 else (
                1 - params["p_slip"][c]
            )
            params["p_slip"][c] = np.clip(1 - correct_given_know, 1e-3, 1 - 1e-3)
            params["p_guess"][c] = np.clip(
                guess_num[c] / guess_denom[c] if guess_denom[c] > 0 else params["p_guess"][c],
                1e-3,
                1 - 1e-3,
            )

        history.append(total_loglik)
        if it > 0 and abs(history[-1] - history[-2]) < tol * abs(history[-2]):
            break

    return params, history


em_sequences = _build_sequences(df_sorted)
BKT_PARAMS_FITTED, em_history = fit_bkt_em(em_sequences, BKT_PARAMS, n_iter=30)

print(f"EM converged in {len(em_history)} iterations, final loglik: {em_history[-1]:.2f}")
print("\nfitted params:")
for k, v in BKT_PARAMS_FITTED.items():
    print(f"  {k}: {v}")
print("\nhand-picked params (for comparison):")
for k, v in BKT_PARAMS.items():
    print(f"  {k}: {v}")

EM converged in 14 iterations, final loglik: -6711.56

fitted params:
  p_init: 0.4032570685978337
  p_transit: 0.055144557674882626
  p_slip: {'confident': np.float64(0.17708832341950154), 'unsure': np.float64(0.4189207790951278), 'guessing': np.float64(0.5346107896636811)}
  p_guess: {'confident': np.float64(0.553197977414209), 'unsure': np.float64(0.4873556642540139), 'guessing': np.float64(0.2299548620238852)}

hand-picked params (for comparison):
  p_init: 0.3
  p_transit: 0.1
  p_slip: {'confident': 0.05, 'unsure': 0.1, 'guessing': 0.3}
  p_guess: {'confident': 0.1, 'unsure': 0.25, 'guessing': 0.5}


In [8]:
bkt_results_fitted, bkt_final_state_fitted = run_bkt(df_sorted, params=BKT_PARAMS_FITTED)

bkt_auc_fitted = roc_auc_score(bkt_results_fitted["correct"], bkt_results_fitted["bkt_pred"])
bkt_ll_fitted = log_loss(
    bkt_results_fitted["correct"], bkt_results_fitted["bkt_pred"].clip(1e-4, 1 - 1e-4)
)
print(f"hand-picked params — AUC: {bkt_auc:.4f}, log loss: {bkt_ll:.4f}")
print(f"EM-fitted params  — AUC: {bkt_auc_fitted:.4f}, log loss: {bkt_ll_fitted:.4f}")

hand-picked params — AUC: 0.5658, log loss: 0.7491
EM-fitted params  — AUC: 0.6819, log loss: 0.6404


EM-fitted params lift AUC 0.5658 → 0.6819, log loss 0.7491 → 0.6404 — the hand-picked
defaults were the ceiling, not BKT itself.

One fitted value worth flagging, not silently trusting: `p_guess["confident"]` (0.55)
comes out *higher* than `p_guess["guessing"]` (0.23) — inverted from the hand-picked
intuition that a confident answer should rarely be a lucky guess. This isn't a fit bug;
it's EM reporting that the two hidden states aren't fully separable from confidence
alone here. The `confident` bucket runs high accuracy overall (cell 8: 0.71), so when
the model is in the "doesn't know" state during a `confident`-labeled attempt, it's
still often right — pushing `p_guess["confident"]` up. Interpret the fitted `p_slip`/
`p_guess` values as what best explains the observed correctness sequences, not as
a re-confirmation of the original confidence-conditional design intuition.

From here, `run_bkt`/`weakest_topics` downstream can swap in `BKT_PARAMS_FITTED` in
place of the hand-picked `BKT_PARAMS` for the per-student mastery view in section 3.

### Persisting the fit

`BKT_PARAMS_FITTED` only lives in kernel memory otherwise — gone on restart. Write it to
`notebooks/bkt_params_fitted.json` (with the fit metadata) so the latest fit survives,
is diffable in git, and is what a future `src/` integration would load rather than
re-fitting on every request. See [docs/09-knowledge-tracing.md § Keeping params
fresh](../docs/09-knowledge-tracing.md#keeping-params-fresh) for when to re-run this
notebook and commit the updated file.

In [9]:
import json
from datetime import datetime, timezone

BKT_PARAMS_PATH = PROJECT_ROOT / "notebooks" / "bkt_params_fitted.json"

params_out = {
    "fitted_at": datetime.now(timezone.utc).isoformat(),
    "n_attempts": len(df_sorted),
    "n_students": df_sorted["student_id"].nunique(),
    "n_topics": df_sorted["topic_path"].nunique(),
    "em_iterations": len(em_history),
    "final_loglik": em_history[-1],
    "auc": bkt_auc_fitted,
    "log_loss": bkt_ll_fitted,
    "params": BKT_PARAMS_FITTED,
}
with open(BKT_PARAMS_PATH, "w") as f:
    json.dump(params_out, f, indent=2, default=float)

print(f"wrote fitted params to {BKT_PARAMS_PATH}")

wrote fitted params to /Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/notebooks/bkt_params_fitted.json


## 3. Per-student weak-topic view

This is the shape a `MASTERS` edge (`Student -> Topic`, `p_know`) would take in the graph
— see the design note at the end for wiring this into `src/quiz/attempts.py::record_attempt`.

Each student gets their own ranked list — this is the personalization output: not one
global model output, but one `p_know` value per (student, topic) pair, so weak topics are
specific to that student's own history and never averaged across the cohort.

In [10]:
mastery_rows = [
    {"student_id": sid, "topic_path": tp, "p_know": pk}
    for (sid, tp), pk in bkt_final_state_fitted.items()
]
mastery_df = pd.DataFrame(mastery_rows)

n_observations = (
    df_sorted.groupby(["student_id", "topic_path"]).size().rename("n_observations")
)
mastery_df = mastery_df.join(n_observations, on=["student_id", "topic_path"])

# attempt_count (from the :REVIEWING edge, src/quiz/attempts.py::record_attempt) is a
# lifetime per-question retry counter — it never resets, unlike streak. Averaged per
# (student, topic) it flags a pattern plain accuracy/n_observations can't: a topic where
# the student keeps re-attempting the same questions and still isn't landing a strong
# pass reads as "stuck", not just "weak" or "under-observed".
avg_attempt_count = (
    df_sorted.groupby(["student_id", "topic_path"])["attempt_count"]
    .mean()
    .rename("avg_attempt_count")
)
mastery_df = mastery_df.join(avg_attempt_count, on=["student_id", "topic_path"])
mastery_df.sort_values(["student_id", "p_know"]).head(10)

,student_id,topic_path,p_know,n_observations,avg_attempt_count
10,088a0ba8-56d7-44ed-9116-79ec03fd3453,ENZIME FK 02 2024 rc > Enzyme isozymes classif...,0.199041,9,1.666667
34,088a0ba8-56d7-44ed-9116-79ec03fd3453,Thyroid > Thyroid hormone synthesis,0.205576,8,3.250000
27,088a0ba8-56d7-44ed-9116-79ec03fd3453,Gi Track Ii Rev 2026 > Intestinal histology st...,0.213826,6,1.000000
35,088a0ba8-56d7-44ed-9116-79ec03fd3453,karbohidrat > Carbohydrate digestion enzymes,0.225869,6,1.666667
5,088a0ba8-56d7-44ed-9116-79ec03fd3453,Anatomy of Neck > Anatomy of Neck,0.266354,14,2.142857
6,088a0ba8-56d7-44ed-9116-79ec03fd3453,Anatomy of Neck > Cervical nerve anatomy,0.331300,6,2.000000
11,088a0ba8-56d7-44ed-9116-79ec03fd3453,ENZIME FK 02 2024 rc > Kinetika Enzim Michaeli...,0.383973,6,2.333333
13,088a0ba8-56d7-44ed-9116-79ec03fd3453,Embriologi Pencernaan > Embryology of the dige...,0.397250,6,3.000000
18,088a0ba8-56d7-44ed-9116-79ec03fd3453,Endocrine 1-25 > Antidiuretic hormone regulation,0.403897,9,1.444444
3,088a0ba8-56d7-44ed-9116-79ec03fd3453,Anatomy of Endocrine Glands > Ovarian blood su...,0.434125,6,1.666667


In [11]:
def weakest_topics(
    student_id: str,
    mastery_df: pd.DataFrame,
    top_n: int = 5,
    min_observations: int = 3,
    stuck_attempt_threshold: float = 2.0,
) -> pd.DataFrame:
    """This student's lowest-p_know topics, ascending — the recommend-what-to-study query
    the personalization system runs (GET /students/me/mastery, sorted client- or
    server-side).

    Adds a `low_evidence` flag for pairs with fewer than `min_observations` attempts: with
    only 1-2 attempts, p_know is still close to p_init (the prior) rather than a real
    read on the student, so surfacing it as "weak" without qualification would overstate
    confidence. Rows are NOT dropped — a topic never attempted is still worth surfacing as
    "unknown, go try it" — just labeled so the caller (or UI) can render it differently
    (e.g. "not enough data yet" instead of a confident weak-topic claim).

    Also adds a `stuck` flag for pairs with enough evidence (not low_evidence) whose
    avg_attempt_count is at least `stuck_attempt_threshold` — the student has repeatedly
    re-attempted these questions (not just answered once and moved on) and still hasn't
    built up p_know, which reads differently from a topic that's merely weak on first
    exposure and worth calling out separately in the UI (e.g. "revisit with a different
    approach" instead of "just practice more")."""
    out = (
        mastery_df[mastery_df["student_id"] == student_id]
        .sort_values("p_know")
        .head(top_n)
        .reset_index(drop=True)
    )
    out["low_evidence"] = out["n_observations"] < min_observations
    out["stuck"] = (~out["low_evidence"]) & (
        out["avg_attempt_count"] >= stuck_attempt_threshold
    )
    return out


example_student = mastery_df["student_id"].iloc[0]
print(f"weakest topics for student {example_student}:")
weakest_topics(example_student, mastery_df)

weakest topics for student 088a0ba8-56d7-44ed-9116-79ec03fd3453:


,student_id,topic_path,p_know,n_observations,avg_attempt_count,low_evidence,stuck
0,088a0ba8-56d7-44ed-9116-79ec03fd3453,ENZIME FK 02 2024 rc > Enzyme isozymes classif...,0.199041,9,1.666667,False,False
1,088a0ba8-56d7-44ed-9116-79ec03fd3453,Thyroid > Thyroid hormone synthesis,0.205576,8,3.250000,False,True
2,088a0ba8-56d7-44ed-9116-79ec03fd3453,Gi Track Ii Rev 2026 > Intestinal histology st...,0.213826,6,1.000000,False,False
3,088a0ba8-56d7-44ed-9116-79ec03fd3453,karbohidrat > Carbohydrate digestion enzymes,0.225869,6,1.666667,False,False
4,088a0ba8-56d7-44ed-9116-79ec03fd3453,Anatomy of Neck > Anatomy of Neck,0.266354,14,2.142857,False,True


In [12]:
sid = example_student
weak = weakest_topics(sid, mastery_df, top_n=3)
for tp in weak["topic_path"]:
    sub = df_sorted[(df_sorted.student_id == sid) & (df_sorted.topic_path == tp)]
    print(tp, "acc:", sub["correct"].mean(), "n:", len(sub))

ENZIME FK 02 2024 rc > Enzyme isozymes classification acc: 0.0 n: 9
Thyroid > Thyroid hormone synthesis acc: 0.0 n: 8
Gi Track Ii Rev 2026 > Intestinal histology structure acc: 0.16666666666666666 n: 6
